# Part 2 | Session 8: Scaling Law - 최적의 데이터 규모와 모델 크기

**© Copyright AIDENTIFY. All rights reserved.**

## 학습 목표

- Scaling Law의 핵심 개념과 역사적 발전 과정을 이해한다.
- Loss와 모델 크기(N), 데이터량(D), 연산량(C)의 관계식을 학습하고 시각화한다.
- Chinchilla Optimal 법칙을 통해 모델 크기와 데이터량의 최적 비율을 파악한다.
- 주어진 compute budget에서 최적의 모델 크기를 계산하는 실무 능력을 기른다.

In [ ]:
# 필요한 패키지 설치
!pip install -q matplotlib numpy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# 한글 폰트 설정 (Colab/Linux 환경)
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print("라이브러리 로드 완료")

---
## 1️⃣ Scaling Law 개요

### Scaling Law란?

Scaling Law는 **모델 크기(N)**, **학습 데이터량(D)**, **연산량(C)**이 증가함에 따라 모델의 성능(loss)이 어떻게 변화하는지를 설명하는 경험적 법칙입니다.

### 주요 연구

| 논문 | 연도 | 핵심 발견 |
|------|------|----------|
| **Kaplan et al.** ("Scaling Laws for Neural Language Models") | 2020 | Loss는 N, D, C 각각에 대해 **power law**를 따름. 모델 크기가 가장 중요한 요소. |
| **Hoffmann et al.** ("Chinchilla", "Training Compute-Optimal Large Language Models") | 2022 | 모델 크기와 데이터량을 **균등하게** 스케일링해야 최적. 기존 모델들은 대부분 undertrained. |

### Kaplan et al. (2020) 핵심 주장

- 성능은 모델 크기(파라미터 수)에 가장 민감하게 반응
- 모델 구조(depth vs width)의 영향은 상대적으로 작음
- 큰 모델을 적은 데이터로 학습시키는 것이 효율적이라고 주장

### Chinchilla (Hoffmann et al., 2022) 반박

- Kaplan의 결론을 수정: **모델 크기와 데이터를 동일한 비율로 증가**시켜야 최적
- 동일 compute에서 Gopher(280B)보다 작은 Chinchilla(70B)가 **4배 많은 데이터**로 학습하여 더 우수한 성능 달성
- 최적 비율: 파라미터 수 대비 약 **20배**의 학습 토큰 필요

---
## 2️⃣ Loss = f(N, D, C) 관계식 설명 및 시각화

### Kaplan Scaling Law 관계식

Kaplan et al.은 다음과 같은 power-law 관계를 발견했습니다:

$$L(N) = \left(\frac{N_c}{N}\right)^{\alpha_N}, \quad \alpha_N \approx 0.076, \quad N_c \approx 8.8 \times 10^{13}$$

$$L(D) = \left(\frac{D_c}{D}\right)^{\alpha_D}, \quad \alpha_D \approx 0.095, \quad D_c \approx 5.4 \times 10^{13}$$

$$L(C) = \left(\frac{C_c}{C}\right)^{\alpha_C}, \quad \alpha_C \approx 0.050, \quad C_c \approx 3.1 \times 10^{8}$$

여기서 $N$은 파라미터 수, $D$는 데이터 토큰 수, $C$는 compute(FLOPs)입니다.

In [ ]:
# Kaplan Scaling Law 시각화: Loss vs N, D, C

# 파라미터 설정 (Kaplan et al. 근사값)
Nc = 8.8e13
alpha_N = 0.076

Dc = 5.4e13
alpha_D = 0.095

Cc = 3.1e8
alpha_C = 0.050

# 범위 설정
N_range = np.logspace(6, 12, 200)   # 1M ~ 1T 파라미터
D_range = np.logspace(7, 13, 200)   # 10M ~ 10T 토큰
C_range = np.logspace(15, 25, 200)  # PetaFLOPs 범위

# Loss 계산
L_N = (Nc / N_range) ** alpha_N
L_D = (Dc / D_range) ** alpha_D
L_C = (Cc / C_range) ** alpha_C

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss vs N
axes[0].loglog(N_range, L_N, 'b-', linewidth=2)
axes[0].set_xlabel('Parameters (N)', fontsize=12)
axes[0].set_ylabel('Test Loss', fontsize=12)
axes[0].set_title('Loss vs Model Size (N)', fontsize=14)
axes[0].grid(True, alpha=0.3)

# Loss vs D
axes[1].loglog(D_range, L_D, 'r-', linewidth=2)
axes[1].set_xlabel('Tokens (D)', fontsize=12)
axes[1].set_ylabel('Test Loss', fontsize=12)
axes[1].set_title('Loss vs Dataset Size (D)', fontsize=14)
axes[1].grid(True, alpha=0.3)

# Loss vs C
axes[2].loglog(C_range, L_C, 'g-', linewidth=2)
axes[2].set_xlabel('Compute (FLOPs)', fontsize=12)
axes[2].set_ylabel('Test Loss', fontsize=12)
axes[2].set_title('Loss vs Compute (C)', fontsize=14)
axes[2].grid(True, alpha=0.3)

plt.suptitle('Kaplan Scaling Laws: Power-Law Relationships', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

print("=> 세 그래프 모두 log-log 스케일에서 직선 형태를 보입니다.")
print("=> 이는 Loss가 N, D, C에 대해 power-law를 따른다는 것을 의미합니다.")

In [ ]:
# 결합 Scaling Law: L(N, D) = E + A/N^alpha + B/D^beta
# Chinchilla 논문의 3번째 모델링 접근법 (parametric fit)

E = 1.69       # irreducible loss (엔트로피 하한)
A = 406.4
B = 410.7
alpha = 0.34
beta = 0.28

def combined_loss(N, D):
    """Chinchilla parametric loss model"""
    return E + A / (N ** alpha) + B / (D ** beta)

# N과 D를 변화시키면서 Loss 등고선 그리기
N_vals = np.logspace(7, 11, 100)   # 10M ~ 100B params
D_vals = np.logspace(8, 13, 100)   # 100M ~ 10T tokens

N_grid, D_grid = np.meshgrid(N_vals, D_vals)
L_grid = combined_loss(N_grid, D_grid)

fig, ax = plt.subplots(figsize=(10, 7))
contour = ax.contourf(np.log10(N_grid), np.log10(D_grid), L_grid,
                       levels=30, cmap='viridis_r')
plt.colorbar(contour, ax=ax, label='Loss L(N, D)')

ax.set_xlabel('log10(Parameters N)', fontsize=12)
ax.set_ylabel('log10(Tokens D)', fontsize=12)
ax.set_title('Combined Loss L(N, D) = E + A/N^a + B/D^b', fontsize=14)

# Chinchilla optimal line (D ≈ 20 * N)
N_opt = np.logspace(7, 11, 50)
D_opt = 20 * N_opt
ax.plot(np.log10(N_opt), np.log10(D_opt), 'r--', linewidth=2, label='Chinchilla Optimal (D=20N)')
ax.legend(fontsize=11)

plt.tight_layout()
plt.show()

print("=> 빨간 점선이 Chinchilla Optimal 경로입니다.")
print("=> 동일 compute에서 이 경로를 따라야 loss가 최소화됩니다.")

---
## 3️⃣ Chinchilla Optimal: 모델 크기 vs 데이터량 Tradeoff

### 핵심 결론

Chinchilla 논문의 가장 중요한 결론:

> **Compute budget $C$가 주어졌을 때, 최적 모델 크기 $N_{opt}$와 최적 데이터량 $D_{opt}$는 $C$에 비례하여 동일한 비율로 증가해야 한다.**

$$N_{opt} \propto C^{0.50}, \quad D_{opt} \propto C^{0.50}$$

$$D_{opt} \approx 20 \times N_{opt}$$

즉, 1B 파라미터 모델의 최적 학습 데이터량은 약 **20B 토큰**입니다.

### 실제 모델들은 얼마나 최적에 가까웠을까?

In [ ]:
# 실제 모델들의 파라미터 수 vs 학습 토큰 수 비교

models = {
    'GPT-3':        {'N': 175e9,  'D': 300e9,   'color': 'red',    'marker': 's'},
    'Gopher':       {'N': 280e9,  'D': 300e9,   'color': 'orange', 'marker': 's'},
    'Chinchilla':   {'N': 70e9,   'D': 1400e9,  'color': 'green',  'marker': '*'},
    'LLaMA-7B':     {'N': 7e9,    'D': 1000e9,  'color': 'blue',   'marker': '^'},
    'LLaMA-13B':    {'N': 13e9,   'D': 1000e9,  'color': 'blue',   'marker': '^'},
    'LLaMA-65B':    {'N': 65e9,   'D': 1400e9,  'color': 'blue',   'marker': '^'},
    'LLaMA-2-70B':  {'N': 70e9,   'D': 2000e9,  'color': 'purple', 'marker': 'D'},
    'Mistral-7B':   {'N': 7e9,    'D': 8000e9,  'color': 'cyan',   'marker': 'o'},
}

fig, ax = plt.subplots(figsize=(12, 8))

for name, info in models.items():
    ratio = info['D'] / info['N']
    ax.scatter(info['N'], info['D'], c=info['color'], marker=info['marker'],
               s=200, zorder=5, edgecolors='black', linewidth=0.5)
    ax.annotate(f"{name}\n(D/N={ratio:.0f})",
                (info['N'], info['D']),
                textcoords='offset points', xytext=(10, 5), fontsize=9)

# Chinchilla optimal line (D = 20N)
N_line = np.logspace(9, 12, 100)
D_line_20 = 20 * N_line
ax.plot(N_line, D_line_20, 'r--', linewidth=2, alpha=0.7, label='Chinchilla Optimal (D=20N)')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Parameters (N)', fontsize=13)
ax.set_ylabel('Training Tokens (D)', fontsize=13)
ax.set_title('Model Size vs Training Tokens: Chinchilla Optimality', fontsize=15)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n[분석 결과]")
print("- GPT-3, Gopher: D/N ≈ 1.7, 1.1 → Chinchilla 기준으로 심하게 undertrained")
print("- Chinchilla: D/N = 20 → 최적에 근접")
print("- LLaMA 시리즈: D/N > 20 → overtrained (의도적으로 추론 효율성 추구)")
print("- Mistral-7B: D/N > 1000 → 극도로 overtrained (작은 모델의 성능 극대화 전략)")

### Chinchilla 이후의 트렌드 변화

Chinchilla 논문 이후, 업계의 전략이 변화했습니다:

| 전략 | 설명 | 대표 모델 |
|------|------|----------|
| **Chinchilla Optimal** | D/N ≈ 20, 학습 compute 최소화 | Chinchilla (70B) |
| **Inference Optimal** | D/N >> 20, 작은 모델에 많은 데이터 | LLaMA, Mistral |

**왜 Chinchilla Optimal을 넘어서 더 많은 데이터를 사용할까?**

- 학습 비용은 **한 번**만 들지만, 추론 비용은 **서빙할 때마다** 발생
- 작은 모델이 추론 시 더 빠르고 저렴
- 따라서 학습에 더 많은 compute를 투자하더라도, 작은 모델을 충분히 학습시키는 것이 **총 비용** 면에서 유리

---
## 4️⃣ 실무 적용: Compute Budget에서 최적 모델 크기 계산

### Compute 계산 공식

Transformer 모델의 학습에 필요한 compute (FLOPs):

$$C \approx 6 \times N \times D$$

여기서:
- $C$: 총 FLOPs
- $N$: 파라미터 수
- $D$: 학습 토큰 수
- 6: forward(2) + backward(4) pass의 곱셈 연산

In [ ]:
def compute_budget(N, D):
    """학습에 필요한 총 FLOPs 계산 (C ≈ 6ND)"""
    return 6 * N * D

def chinchilla_optimal(C):
    """
    주어진 compute budget C에서 Chinchilla optimal N, D 계산
    C = 6ND, D = 20N  =>  C = 120 * N^2  =>  N = sqrt(C/120)
    """
    N_opt = np.sqrt(C / 120)
    D_opt = 20 * N_opt
    return N_opt, D_opt

def format_number(n):
    """숫자를 읽기 좋은 형태로 변환"""
    if n >= 1e12:
        return f"{n/1e12:.1f}T"
    elif n >= 1e9:
        return f"{n/1e9:.1f}B"
    elif n >= 1e6:
        return f"{n/1e6:.1f}M"
    else:
        return f"{n:.0f}"

# 다양한 compute budget에서 최적값 계산
print("=" * 75)
print(f"{'Compute (FLOPs)':>20} | {'Optimal N (params)':>20} | {'Optimal D (tokens)':>20}")
print("=" * 75)

compute_budgets = [1e18, 1e19, 1e20, 1e21, 1e22, 1e23, 1e24]

for C in compute_budgets:
    N_opt, D_opt = chinchilla_optimal(C)
    print(f"{format_number(C) + ' FLOPs':>20} | {format_number(N_opt):>20} | {format_number(D_opt):>20}")

print("=" * 75)

In [ ]:
# 실습: GPU 기반 compute budget 계산

# GPU 스펙 (BF16 기준 TFLOPS)
gpu_specs = {
    'A100 (80GB)': 312e12,       # 312 TFLOPS (BF16)
    'H100 (80GB)': 990e12,       # 990 TFLOPS (BF16)
    'H200 (141GB)': 990e12,      # 990 TFLOPS (BF16)
}

# MFU (Model FLOPs Utilization) - 실제로는 30~60% 정도
MFU = 0.40  # 40% 활용률 가정

print("[시나리오] 주어진 GPU 자원으로 학습 가능한 최적 모델 크기\n")

# 시나리오: 64x H100, 30일 학습
num_gpus = 64
gpu_type = 'H100 (80GB)'
training_days = 30
training_seconds = training_days * 24 * 3600

# 총 compute budget
total_flops = num_gpus * gpu_specs[gpu_type] * MFU * training_seconds

print(f"GPU: {num_gpus}x {gpu_type}")
print(f"Training duration: {training_days} days")
print(f"MFU: {MFU*100:.0f}%")
print(f"Total compute: {total_flops:.2e} FLOPs")
print()

# Chinchilla optimal
N_opt, D_opt = chinchilla_optimal(total_flops)
print(f"[Chinchilla Optimal]")
print(f"  Model size: {format_number(N_opt)} parameters")
print(f"  Training data: {format_number(D_opt)} tokens")
print()

# Inference optimal (D = 200N, 더 작은 모델에 더 많은 데이터)
# C = 6ND, D = 200N => C = 1200 * N^2
N_inf = np.sqrt(total_flops / 1200)
D_inf = 200 * N_inf
print(f"[Inference Optimal (D=200N)]")
print(f"  Model size: {format_number(N_inf)} parameters")
print(f"  Training data: {format_number(D_inf)} tokens")

---
## 정리

### 핵심 요약

1. **Scaling Law**는 모델 성능(loss)이 N, D, C에 대해 power-law를 따른다는 경험적 법칙
2. **Kaplan (2020)**: 모델 크기가 가장 중요 → 큰 모델을 적게 학습
3. **Chinchilla (2022)**: N과 D를 균등하게 스케일링 → D ≈ 20N이 최적
4. **실무에서는**: 추론 비용까지 고려하여 Chinchilla optimal보다 더 많은 데이터로 학습하는 추세 (LLaMA, Mistral)
5. **Compute 계산**: C ≈ 6ND 공식으로 필요한 자원을 추정 가능

### 참고 문헌

- Kaplan et al. (2020). "Scaling Laws for Neural Language Models." [arXiv:2001.08361](https://arxiv.org/abs/2001.08361)
- Hoffmann et al. (2022). "Training Compute-Optimal Large Language Models." [arXiv:2203.15556](https://arxiv.org/abs/2203.15556)
- Touvron et al. (2023). "LLaMA: Open and Efficient Foundation Language Models." [arXiv:2302.13971](https://arxiv.org/abs/2302.13971)